# 23-09 · Сканируем настоящий каталог

Практика к разделу [«Сканируем каталог»](../../site/chapters/glava-23/23-08-skaniruem-katalog.html). Использует настоящий пакет `safesort` (`projects/python/safesort/`).

## Цель

Прогнать настоящую функцию `safesort.scanner.scan()` по временному каталогу с вложенными файлами и убедиться, что она находит именно то, что нужно, — и ничего лишнего.

## Рабочий пример

In [1]:
import tempfile
from pathlib import Path

from safesort.config import Config
from safesort.scanner import scan

tmpdir = tempfile.TemporaryDirectory()
koren = Path(tmpdir.name)

(koren / "podkatalog").mkdir()
(koren / "podkatalog" / "otchet.pdf").write_text("...", encoding="utf-8")
(koren / "photo.jpg").write_text("...", encoding="utf-8")
(koren / ".git").mkdir()
(koren / ".git" / "config").write_text("...", encoding="utf-8")

fajly = scan(koren, Config())
imena = {f.path.name for f in fajly}
print("Найдено файлов:", len(fajly))
print(imena)

Найдено файлов: 2
{'otchet.pdf', 'photo.jpg'}


## Проверка результата

In [2]:
assert imena == {"otchet.pdf", "photo.jpg"}
assert all(f.path.name != "config" for f in fajly)  # .git исключён по умолчанию
print("Верно: сканер нашёл вложенный файл и файл в корне, но не заглянул в .git.")

Верно: сканер нашёл вложенный файл и файл в корне, но не заглянул в .git.


## Эксперимент — повторный запуск не находит уже отсортированные файлы

In [3]:
(koren / "Sorted" / "documents").mkdir(parents=True)
(koren / "Sorted" / "documents" / "staryj.pdf").write_text("...", encoding="utf-8")

fajly_posle = scan(koren, Config())
imena_posle = {f.path.name for f in fajly_posle}

assert "staryj.pdf" not in imena_posle
assert imena_posle == {"otchet.pdf", "photo.jpg"}
print("Верно: каталог результата Sorted/ исключён из повторного сканирования.")

Верно: каталог результата Sorted/ исключён из повторного сканирования.


## Задание ★★ Самостоятельная задача

Добавьте символическую ссылку на `photo.jpg` и убедитесь, что сканер её пропускает.

In [4]:
ssylka = koren / "ssylka_na_foto.jpg"
try:
    ssylka.symlink_to(koren / "photo.jpg")
    podderzhivayutsya_ssylki = True
except (OSError, NotImplementedError):
    podderzhivayutsya_ssylki = False

if podderzhivayutsya_ssylki:
    fajly_so_ssylkoj = scan(koren, Config())
    imena_so_ssylkoj = {f.path.name for f in fajly_so_ssylkoj}
    assert "ssylka_na_foto.jpg" not in imena_so_ssylkoj
    print("Верно: символическая ссылка не попала в список найденных файлов.")
else:
    print("Символические ссылки не поддерживаются в этом окружении — пропускаем эту проверку.")

tmpdir.cleanup()

Верно: символическая ссылка не попала в список найденных файлов.
